In [3]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Seed fisso: risultati riproducibili
random.seed(42)
np.random.seed(42)

# I nostri clienti finti
clienti = [
    "Rossi Costruzioni SRL", "Bianchi Alimentari SPA", "Verdi Logistica SRL",
    "Toscana Impianti SNC", "Meccanica Fiorentina SRL", "Studio Neri & Associati",
    "Elettrodomestici Gallo SRL", "Tessuti Lucchesi SPA", "AgriToscana SOC COOP",
    "InfoService SRL"
]

# Generiamo 60 fatture nel primo semestre 2026
fatture = []
for i in range(1, 61):
    data_emissione = datetime(2026, 1, 1) + timedelta(days=random.randint(0, 150))
    fatture.append({
        "numero_fattura": f"FT-2026-{i:03d}",
        "cliente": random.choice(clienti),
        "importo": round(random.uniform(200, 5000), 2),
        "data_emissione": data_emissione.date(),
        "data_scadenza": (data_emissione + timedelta(days=30)).date()
    })

df_fatture = pd.DataFrame(fatture)
df_fatture.head(10)

,numero_fattura,cliente,importo,data_emissione,data_scadenza
0,FT-2026-001,Rossi Costruzioni SRL,3759.44,2026-01-29,2026-02-28
1,FT-2026-002,Toscana Impianti SNC,869.78,2026-03-04,2026-04-03
2,FT-2026-003,AgriToscana SOC COOP,617.31,2026-01-27,2026-02-26
3,FT-2026-004,Rossi Costruzioni SRL,343.03,2026-04-19,2026-05-19
4,FT-2026-005,Toscana Impianti SNC,2625.71,2026-02-25,2026-03-27
5,FT-2026-006,AgriToscana SOC COOP,1154.42,2026-01-07,2026-02-06
6,FT-2026-007,Elettrodomestici Gallo SRL,1258.11,2026-05-20,2026-06-19
7,FT-2026-008,Meccanica Fiorentina SRL,4085.27,2026-05-31,2026-06-30
8,FT-2026-009,Verdi Logistica SRL,3551.07,2026-01-02,2026-02-01
9,FT-2026-010,Meccanica Fiorentina SRL,946.30,2026-03-29,2026-04-28


In [4]:
pip install pandas numpy


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
movimenti = []

def sporca_causale(numero_fattura, cliente):
    """Genera una causale realistica, scritta in uno dei tanti modi possibili"""
    n = int(numero_fattura.split("-")[2])
    stili = [
        f"PAGAMENTO {numero_fattura}",
        f"FT {n}",
        f"SALDO FATT. {n}/2026 {cliente.upper()}",
        f"BONIFICO DA {cliente.upper()}",
        "PAGAMENTO FORNITURA",
    ]
    return random.choice(stili)

# --- DESTINI ---
indici = list(df_fatture.index)
random.shuffle(indici)
non_pagate = indici[:8]  # 8 fatture restano non pagate

# --- BONIFICI CUMULATIVI (prima dei singoli!) ---
# 2 clienti pagano 3 delle LORO fatture con un solo bonifico
clienti_cumulativi = ["Meccanica Fiorentina SRL", "Elettrodomestici Gallo SRL"]
indici_cumulativi = []

for cliente in clienti_cumulativi:
    fatture_cliente = df_fatture[
        (df_fatture["cliente"] == cliente) &
        (~df_fatture.index.isin(non_pagate)) &
        (~df_fatture.index.isin(indici_cumulativi))
    ].head(3)

    if len(fatture_cliente) < 2:
        continue  # troppe poche fatture per un cumulativo: salto

    indici_cumulativi.extend(fatture_cliente.index)
    movimenti.append({
        "data": fatture_cliente["data_scadenza"].max() + timedelta(days=random.randint(0, 10)),
        "importo": round(fatture_cliente["importo"].sum(), 2),
        "causale": f"SALDO FATTURE MESE {cliente.upper()}"
    })

# --- PAGAMENTI SINGOLI ---
for i in indici[8:]:  # tutti tranne le non pagate
    if i in indici_cumulativi:
        continue  # già pagate nel bonifico cumulativo
    f = df_fatture.loc[i]
    ritardo = random.randint(-2, 20)
    importo = f["importo"]
    if random.random() < 0.15:  # 15% dei casi: commissione trattenuta
        importo = round(importo - 2.50, 2)
    movimenti.append({
        "data": f["data_scadenza"] + timedelta(days=ritardo),
        "importo": importo,
        "causale": sporca_causale(f["numero_fattura"], f["cliente"])
    })

# --- RUMORE ---
rumore = [
    {"data": datetime(2026, 2, 16).date(), "importo": -1850.00, "causale": "F24 TASSE E CONTRIBUTI"},
    {"data": datetime(2026, 3, 27).date(), "importo": -4200.00, "causale": "STIPENDI FEBBRAIO"},
    {"data": datetime(2026, 3, 31).date(), "importo": -12.50, "causale": "COMMISSIONI TENUTA CONTO"},
    {"data": datetime(2026, 4, 15).date(), "importo": 500.00, "causale": "GIROCONTO INTERNO"},
]
movimenti.extend(rumore)

df_movimenti = pd.DataFrame(movimenti).sort_values("data").reset_index(drop=True)
print(f"Movimenti totali: {len(df_movimenti)}")
df_movimenti.head(10)

Movimenti totali: 53


,data,importo,causale
0,2026-02-01,3472.21,PAGAMENTO FT-2026-040
1,2026-02-01,3551.07,SALDO FATT. 9/2026 VERDI LOGISTICA SRL
2,2026-02-14,4144.65,PAGAMENTO FT-2026-026
3,2026-02-16,-1850.00,F24 TASSE E CONTRIBUTI
4,2026-02-25,4116.91,SALDO FATT. 52/2026 TESSUTI LUCCHESI SPA
5,2026-02-26,617.31,SALDO FATT. 3/2026 AGRITOSCANA SOC COOP
6,2026-03-02,4333.29,PAGAMENTO FORNITURA
7,2026-03-04,3247.90,PAGAMENTO FT-2026-022
8,2026-03-05,3775.95,PAGAMENTO FORNITURA
9,2026-03-06,4265.47,PAGAMENTO FORNITURA


In [6]:
df_fatture.to_csv("fatture.csv", index=False)
df_movimenti.to_csv("estratto_conto.csv", index=False)
print("File salvati!")

File salvati!


In [7]:
import re

# Estraiamo il numero progressivo da ogni fattura (es. da FT-2026-035 -> 35)
df_fatture["numero"] = df_fatture["numero_fattura"].str.split("-").str[2].astype(int)

# Dizionario: numero -> riga della fattura (per lookup veloce)
fatture_per_numero = {row["numero"]: idx for idx, row in df_fatture.iterrows()}

risultati = []

for idx, mov in df_movimenti.iterrows():
    match_trovato = None

    # 1. Estraggo TUTTI i numeri dalla causale
    numeri_in_causale = [int(n) for n in re.findall(r"\d+", mov["causale"])]

    # 2. Tengo solo quelli che corrispondono a fatture esistenti
    candidati = [n for n in numeri_in_causale if n in fatture_per_numero]

    # 3. Tra i candidati, cerco quello con importo corrispondente
    for n in candidati:
        fattura = df_fatture.loc[fatture_per_numero[n]]
        if abs(mov["importo"] - fattura["importo"]) < 0.01:
            match_trovato = fattura["numero_fattura"]
            break

    risultati.append({
        "data_movimento": mov["data"],
        "importo": mov["importo"],
        "causale": mov["causale"],
        "fattura_matchata": match_trovato,
        "livello": "1 - esatto" if match_trovato else None
    })

df_risultati = pd.DataFrame(risultati)
matchati = df_risultati["fattura_matchata"].notna().sum()
print(f"Livello 1: matchati {matchati} movimenti su {len(df_risultati)}")
df_risultati[df_risultati["fattura_matchata"].notna()].head(10)

Livello 1: matchati 24 movimenti su 53

,data_movimento,importo,causale,fattura_matchata,livello
0,2026-02-01,3472.21,PAGAMENTO FT-2026-040,FT-2026-040,1 - esatto
1,2026-02-01,3551.07,SALDO FATT. 9/2026 VERDI LOGISTICA SRL,FT-2026-009,1 - esatto
2,2026-02-14,4144.65,PAGAMENTO FT-2026-026,FT-2026-026,1 - esatto
4,2026-02-25,4116.91,SALDO FATT. 52/2026 TESSUTI LUCCHESI SPA,FT-2026-052,1 - esatto
5,2026-02-26,617.31,SALDO FATT. 3/2026 AGRITOSCANA SOC COOP,FT-2026-003,1 - esatto
7,2026-03-04,3247.90,PAGAMENTO FT-2026-022,FT-2026-022,1 - esatto
12,2026-03-19,215.58,PAGAMENTO FT-2026-043,FT-2026-043,1 - esatto
13,2026-03-22,4001.98,FT 36,FT-2026-036,1 - esatto
18,2026-04-10,2115.16,FT 57,FT-2026-057,1 - esatto
20,2026-04-13,3702.71,FT 13,FT-2026-013,1 - esatto


In [8]:
# Fatture già assegnate al livello 1 (non più disponibili)
fatture_usate = set(df_risultati["fattura_matchata"].dropna())

# Lavoriamo solo sui movimenti non matchati e con importo positivo (incassi)
for idx, riga in df_risultati[df_risultati["fattura_matchata"].isna()].iterrows():
    if riga["importo"] <= 0:
        continue  # F24, stipendi ecc: non possono essere incassi di fatture

    # Cerco fatture aperte con lo stesso importo
    candidate = df_fatture[
        (~df_fatture["numero_fattura"].isin(fatture_usate)) &
        (abs(df_fatture["importo"] - riga["importo"]) < 0.01)
    ]

    # Tra queste, tengo solo quelle con data compatibile
    compatibili = []
    for _, fatt in candidate.iterrows():
        giorni = (riga["data_movimento"] - fatt["data_scadenza"]).days
        if -5 <= giorni <= 30:
            compatibili.append(fatt["numero_fattura"])

    # Match SOLO se il candidato è unico
    if len(compatibili) == 1:
        df_risultati.loc[idx, "fattura_matchata"] = compatibili[0]
        df_risultati.loc[idx, "livello"] = "2 - importo e data"
        fatture_usate.add(compatibili[0])

matchati_tot = df_risultati["fattura_matchata"].notna().sum()
print(f"Dopo il livello 2: matchati {matchati_tot} su {len(df_risultati)}")
df_risultati[df_risultati["livello"] == "2 - importo e data"]

Dopo il livello 2: matchati 37 su 53


,data_movimento,importo,causale,fattura_matchata,livello
6,2026-03-02,4333.29,PAGAMENTO FORNITURA,FT-2026-035,2 - importo e data
8,2026-03-05,3775.95,PAGAMENTO FORNITURA,FT-2026-031,2 - importo e data
10,2026-03-07,300.96,BONIFICO DA STUDIO NERI & ASSOCIATI,FT-2026-060,2 - importo e data
14,2026-03-26,2625.71,PAGAMENTO FORNITURA,FT-2026-005,2 - importo e data
23,2026-04-16,869.78,PAGAMENTO FORNITURA,FT-2026-002,2 - importo e data
26,2026-05-04,645.18,PAGAMENTO FORNITURA,FT-2026-011,2 - importo e data
28,2026-05-07,3123.06,PAGAMENTO FORNITURA,FT-2026-046,2 - importo e data
31,2026-05-14,1976.87,BONIFICO DA VERDI LOGISTICA SRL,FT-2026-020,2 - importo e data
38,2026-05-31,343.03,BONIFICO DA ROSSI COSTRUZIONI SRL,FT-2026-004,2 - importo e data
42,2026-06-15,3785.67,PAGAMENTO FORNITURA,FT-2026-032,2 - importo e data


In [9]:
rimasti = df_risultati[df_risultati["fattura_matchata"].isna()]
print(f"Movimenti non matchati: {len(rimasti)}")
rimasti.sort_values("importo")

Movimenti non matchati: 16


,data_movimento,importo,causale,fattura_matchata,livello
15,2026-03-27,-4200.00,STIPENDI FEBBRAIO,NaN,NaN
3,2026-02-16,-1850.00,F24 TASSE E CONTRIBUTI,NaN,NaN
16,2026-03-31,-12.50,COMMISSIONI TENUTA CONTO,NaN,NaN
22,2026-04-15,500.00,GIROCONTO INTERNO,NaN,NaN
41,2026-06-14,2228.56,FT 55,NaN,NaN
47,2026-06-29,2743.27,SALDO FATTURE MESE ELETTRODOMESTICI GALLO SRL,NaN,NaN
36,2026-05-26,2847.30,PAGAMENTO FORNITURA,NaN,NaN
25,2026-04-29,3343.61,PAGAMENTO FORNITURA,NaN,NaN
11,2026-03-15,3756.94,FT 1,NaN,NaN
49,2026-07-02,3886.77,PAGAMENTO FT-2026-041,NaN,NaN


In [10]:
fatture_aperte = df_fatture[~df_fatture["numero_fattura"].isin(fatture_usate)]

for idx, riga in rimasti.iterrows():
    if riga["importo"] <= 0:
        continue
    vicine = fatture_aperte[abs(fatture_aperte["importo"] - riga["importo"]) < 5]
    for _, f in vicine.iterrows():
        diff = round(f["importo"] - riga["importo"], 2)
        print(f"{riga['causale'][:40]:40} | movimento {riga['importo']:>9} | "
              f"{f['numero_fattura']} da {f['importo']:>9} | differenza: {diff}")

PAGAMENTO FORNITURA                      | movimento   4265.47 | FT-2026-012 da   4267.97 | differenza: 2.5
FT 1                                     | movimento   3756.94 | FT-2026-001 da   3759.44 | differenza: 2.5
PAGAMENTO FORNITURA                      | movimento   4742.86 | FT-2026-051 da   4745.36 | differenza: 2.5
FT 28                                    | movimento   4404.06 | FT-2026-028 da   4406.56 | differenza: 2.5
PAGAMENTO FORNITURA                      | movimento   3343.61 | FT-2026-029 da   3346.11 | differenza: 2.5
FT 58                                    | movimento   4515.65 | FT-2026-058 da   4518.15 | differenza: 2.5
PAGAMENTO FORNITURA                      | movimento    2847.3 | FT-2026-015 da    2849.8 | differenza: 2.5
FT 55                                    | movimento   2228.56 | FT-2026-055 da   2231.06 | differenza: 2.5
PAGAMENTO FT-2026-041                    | movimento   3886.77 | FT-2026-041 da   3889.27 | differenza: 2.5
PAGAMENTO FORNITURA         

In [11]:
df_risultati["differenza"] = 0.0  # nuova colonna: scostamento importo

for idx, riga in df_risultati[df_risultati["fattura_matchata"].isna()].iterrows():
    if riga["importo"] <= 0:
        continue

    causale = riga["causale"].upper()
    fatture_aperte = df_fatture[~df_fatture["numero_fattura"].isin(fatture_usate)]

    # Candidate: importo vicino (entro 5 euro) e data compatibile
    for _, fatt in fatture_aperte.iterrows():
        diff = round(fatt["importo"] - riga["importo"], 2)
        if abs(diff) > 5:
            continue
        giorni = (riga["data_movimento"] - fatt["data_scadenza"]).days
        if not (-5 <= giorni <= 30):
            continue

        # PROVA 1: numero fattura presente in causale
        numeri = [int(n) for n in re.findall(r"\d+", causale)]
        prova_numero = fatt["numero"] in numeri

        # PROVA 2: nome cliente presente in causale
        prova_cliente = fatt["cliente"].upper() in causale

        if prova_numero or prova_cliente:
            df_risultati.loc[idx, "fattura_matchata"] = fatt["numero_fattura"]
            df_risultati.loc[idx, "livello"] = "3 - tolleranza con prova"
            df_risultati.loc[idx, "differenza"] = diff
            fatture_usate.add(fatt["numero_fattura"])
            break

matchati_tot = df_risultati["fattura_matchata"].notna().sum()
print(f"Dopo il livello 3: matchati {matchati_tot} su {len(df_risultati)}")
df_risultati[df_risultati["livello"] == "3 - tolleranza con prova"]

Dopo il livello 3: matchati 42 su 53


,data_movimento,importo,causale,fattura_matchata,livello,differenza
11,2026-03-15,3756.94,FT 1,FT-2026-001,3 - tolleranza con prova,2.5
19,2026-04-12,4404.06,FT 28,FT-2026-028,3 - tolleranza con prova,2.5
29,2026-05-07,4515.65,FT 58,FT-2026-058,3 - tolleranza con prova,2.5
41,2026-06-14,2228.56,FT 55,FT-2026-055,3 - tolleranza con prova,2.5
49,2026-07-02,3886.77,PAGAMENTO FT-2026-041,FT-2026-041,3 - tolleranza con prova,2.5


In [12]:
df_risultati["suggerimento"] = None

fatture_aperte = df_fatture[~df_fatture["numero_fattura"].isin(fatture_usate)]

for idx, riga in df_risultati[df_risultati["fattura_matchata"].isna()].iterrows():
    if riga["importo"] <= 0:
        continue

    candidate = []
    for _, fatt in fatture_aperte.iterrows():
        diff = round(fatt["importo"] - riga["importo"], 2)
        giorni = (riga["data_movimento"] - fatt["data_scadenza"]).days
        if abs(diff) <= 5 and -5 <= giorni <= 30:
            candidate.append(f"{fatt['numero_fattura']} (diff {diff}€)")

    # Suggerisco SOLO se il candidato è unico — coerenza col principio del livello 2
    if len(candidate) == 1:
        df_risultati.loc[idx, "suggerimento"] = candidate[0]

con_suggerimento = df_risultati["suggerimento"].notna().sum()
print(f"Eccezioni con suggerimento: {con_suggerimento}")
df_risultati[df_risultati["suggerimento"].notna()][["data_movimento", "importo", "causale", "suggerimento"]]

Eccezioni con suggerimento: 5


,data_movimento,importo,causale,suggerimento
9,2026-03-06,4265.47,PAGAMENTO FORNITURA,FT-2026-012 (diff 2.5€)
17,2026-04-05,4742.86,PAGAMENTO FORNITURA,FT-2026-051 (diff 2.5€)
25,2026-04-29,3343.61,PAGAMENTO FORNITURA,FT-2026-029 (diff 2.5€)
36,2026-05-26,2847.30,PAGAMENTO FORNITURA,FT-2026-015 (diff 2.5€)
50,2026-07-05,4606.27,PAGAMENTO FORNITURA,FT-2026-048 (diff 2.5€)


In [13]:
from itertools import combinations

for idx, riga in df_risultati[df_risultati["fattura_matchata"].isna()].iterrows():
    if riga["importo"] <= 0 or riga["suggerimento"] is not None:
        continue

    causale = riga["causale"].upper()

    # 1. Identifico il cliente dalla causale
    cliente_trovato = None
    for c in df_fatture["cliente"].unique():
        if c.upper() in causale:
            cliente_trovato = c
            break
    if cliente_trovato is None:
        continue

    # 2. Fatture aperte di QUEL cliente
    aperte_cliente = df_fatture[
        (df_fatture["cliente"] == cliente_trovato) &
        (~df_fatture["numero_fattura"].isin(fatture_usate))
    ]

    # 3. Provo combinazioni di 2 e di 3 fatture
    match_combo = None
    for k in [2, 3]:
        for combo in combinations(aperte_cliente.index, k):
            somma = df_fatture.loc[list(combo), "importo"].sum()
            if abs(somma - riga["importo"]) < 0.01:
                match_combo = df_fatture.loc[list(combo), "numero_fattura"].tolist()
                break
        if match_combo:
            break

    if match_combo:
        df_risultati.loc[idx, "fattura_matchata"] = " + ".join(match_combo)
        df_risultati.loc[idx, "livello"] = "4 - bonifico cumulativo"
        fatture_usate.update(match_combo)

matchati_tot = df_risultati["fattura_matchata"].notna().sum()
print(f"Dopo il livello 4: matchati {matchati_tot} su {len(df_risultati)}")
df_risultati[df_risultati["livello"] == "4 - bonifico cumulativo"]

Dopo il livello 4: matchati 44 su 53


,data_movimento,importo,causale,fattura_matchata,livello,differenza,suggerimento
47,2026-06-29,2743.27,SALDO FATTURE MESE ELETTRODOMESTICI GALLO SRL,FT-2026-007 + FT-2026-027,4 - bonifico cumulativo,0.0,None
48,2026-06-30,12603.72,SALDO FATTURE MESE MECCANICA FIORENTINA SRL,FT-2026-008 + FT-2026-021 + FT-2026-024,4 - bonifico cumulativo,0.0,None


In [14]:
# Guardiamo i 2 bonifici cumulativi e le fatture che li compongono
cumulativi = df_risultati[
    df_risultati["fattura_matchata"].isna() &
    df_risultati["causale"].str.contains("SALDO FATTURE MESE")
]

for idx, riga in cumulativi.iterrows():
    print(f"Bonifico: {riga['causale']} | importo {riga['importo']}")

# E ora: quali fatture aperte, sommate, danno quegli importi? Proviamo SENZA vincolo cliente
fatture_aperte = df_fatture[~df_fatture["numero_fattura"].isin(fatture_usate)]
for idx, riga in cumulativi.iterrows():
    for k in [2, 3]:
        for combo in combinations(fatture_aperte.index, k):
            if abs(df_fatture.loc[list(combo), "importo"].sum() - riga["importo"]) < 0.01:
                print(f"\nBonifico da {riga['importo']} = somma di:")

In [15]:
print(df_risultati["livello"].value_counts())
print(f"\nNon matchati: {df_risultati['fattura_matchata'].isna().sum()}")
print(f"Di cui con suggerimento: {df_risultati['suggerimento'].notna().sum()}")

# I match del livello 4 nel dettaglio
df_risultati[df_risultati["livello"] == "4 - bonifico cumulativo"]

livello
1 - esatto                  24
2 - importo e data          13
3 - tolleranza con prova     5
4 - bonifico cumulativo      2
Name: count, dtype: int64

Non matchati: 9
Di cui con suggerimento: 5


,data_movimento,importo,causale,fattura_matchata,livello,differenza,suggerimento
47,2026-06-29,2743.27,SALDO FATTURE MESE ELETTRODOMESTICI GALLO SRL,FT-2026-007 + FT-2026-027,4 - bonifico cumulativo,0.0,None
48,2026-06-30,12603.72,SALDO FATTURE MESE MECCANICA FIORENTINA SRL,FT-2026-008 + FT-2026-021 + FT-2026-024,4 - bonifico cumulativo,0.0,None


In [16]:
# Esporto le fatture NON riconciliate (crediti aperti) per l'analisi crediti
fatture_aperte = df_fatture[~df_fatture["numero_fattura"].isin(fatture_usate)]
fatture_aperte.to_csv("fatture_aperte.csv", index=False)
print(f"Esportate {len(fatture_aperte)} fatture aperte")

Esportate 13 fatture aperte


In [17]:
# Esporto i pagamenti effettuati (per l'analisi comportamentale clienti)
pagamenti = df_risultati[df_risultati["fattura_matchata"].notna()].copy()
pagamenti = pagamenti[~pagamenti["livello"].str.startswith("4")]  # i cumulativi li gestiamo a parte
pagamenti.to_csv("pagamenti_effettuati.csv", index=False)
print(f"Esportati {len(pagamenti)} pagamenti")


Esportati 42 pagamenti
